# Rastreamento da previsão meteorológica

Este notebook consulta a previsão de Bambuí-MG no IPMet Radar.
Os dados são extraídos da tabela HTML, convertidos para valores estruturados e organizados por hora.
Ao final, a tabela é ordenada pela temperatura e exportada para CSV.

In [1]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.ipmetradar.com.br/2prevcid/2prevcid.php?campo=A565&completo=BAMBUI-MG%20(A565)"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://www.ipmetradar.com.br/2prevcid.php",
}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()
response.encoding = "utf-8"
soup = BeautifulSoup(response.text, "html.parser")

# Converte textos meteorológicos para números, tratando '-' como valor ausente.
def numero(valor):
    return pd.to_numeric(
        str(valor).replace(",", ".").replace("°", "").replace("%", "").strip(),
        errors="coerce",
    )


def valores_da_linha(linha):
    return [celula.get_text(" ", strip=True) for celula in linha.find_all("td")]


def encontrar_linha(bloco, classe, texto=None):
    for linha in bloco.find_all("tr", class_=classe):
        if texto is None or texto in linha.get_text(" ", strip=True):
            return linha
    return None


# Algumas linhas ficam fora dos blocos de dia no HTML do site,
# mas mantêm a mesma ordem dos blocos dia1, dia2, ..., dia7.
linhas_precipitacao = soup.find_all("tr", class_="precips")
linhas_probabilidade = [
    linha for linha in soup.find_all("tr", class_="precipprobs")
    if "Probabilidade de chuva" in linha.get_text(" ", strip=True)
]
linhas_nebulosidade = [
    linha for linha in soup.find_all("tr", class_="precipprobs")
    if "Nebulosidade" in linha.get_text(" ", strip=True)
]

registros = []
data_coleta = pd.Timestamp.now().floor("s")

for indice, bloco in enumerate(soup.select("div[id^='dia']")):
    linha_horas = encontrar_linha(bloco, "times")
    linha_data = encontrar_linha(bloco, "icons")
    linha_temperatura = encontrar_linha(bloco, "temperatures")
    linha_umidade = encontrar_linha(bloco, "winddirs", "Umidade Relativa")
    linha_direcao = encontrar_linha(bloco, "winddirs", "Direção do vento")
    linha_velocidade = encontrar_linha(bloco, "windspeeds")
    linha_precipitacao = linhas_precipitacao[indice] if indice < len(linhas_precipitacao) else None
    linha_probabilidade = linhas_probabilidade[indice] if indice < len(linhas_probabilidade) else None
    linha_nebulosidade = linhas_nebulosidade[indice] if indice < len(linhas_nebulosidade) else None

    texto_horas = linha_horas.get_text(" | ", strip=True)
    dia_semana = texto_horas.split("|")[0].strip()
    horas_do_dia = valores_da_linha(linha_horas)
    data_texto = re.search(r"\d{2}/\d{2}", linha_data.get_text(" ", strip=True)).group()
    temperaturas = valores_da_linha(linha_temperatura)
    umidades = valores_da_linha(linha_umidade)
    direcoes = valores_da_linha(linha_direcao)
    precipitacoes = valores_da_linha(linha_precipitacao) if linha_precipitacao else []
    probabilidades = valores_da_linha(linha_probabilidade) if linha_probabilidade else []
    nebulosidades = valores_da_linha(linha_nebulosidade) if linha_nebulosidade else []
    velocidade = numero(valores_da_linha(linha_velocidade)[0])

    for hora, temperatura, umidade, direcao, precipitacao, probabilidade, nebulosidade in zip(
        horas_do_dia,
        temperaturas,
        umidades,
        direcoes,
        precipitacoes,
        probabilidades,
        nebulosidades,
    ):
        registros.append({
            "data_previsao": data_texto,
            "dia_semana": dia_semana,
            "hora": hora,
            "temperatura_c": numero(temperatura),
            "umidade_pct": numero(umidade),
            "direcao_vento": direcao,
            "velocidade_vento_kmh": velocidade,
            "precipitacao_mm_3h": numero(precipitacao),
            "probabilidade_chuva_pct": numero(probabilidade),
            "nebulosidade_pct": numero(nebulosidade),
            "coletado_em": data_coleta,
        })

# DataFrame único usado para exibição, ordenação e exportação.
df_final = pd.DataFrame(registros).sort_values(
    ["data_previsao", "hora"],
    ascending=[True, True],
).reset_index(drop=True)

print("Tabela consolidada:")
print(df_final.to_string(index=False))
print(f"\nRegistros coletados: {len(df_final)}")

df_final.to_csv("previsao_bambui.csv", index=False, encoding="utf-8-sig")
print("Arquivo salvo: previsao_bambui.csv")

Tabela consolidada:
data_previsao    dia_semana hora  temperatura_c  umidade_pct direcao_vento  velocidade_vento_kmh  precipitacao_mm_3h  probabilidade_chuva_pct  nebulosidade_pct         coletado_em
        19/09        Sábado  00h             21           81             L                    16                 NaN                      NaN                 0 2026-09-20 08:26:35
        19/09        Sábado  03h             19           90             L                    16                 0.0                      0.0                 0 2026-09-20 08:26:35
        19/09        Sábado  06h             18           95             L                    16                 0.0                      0.0                 1 2026-09-20 08:26:35
        19/09        Sábado  09h             22           74            NE                    16                 0.0                      0.0                19 2026-09-20 08:26:35
        19/09        Sábado  12h             28           51            NE      

In [7]:
print("Registros por data:")
print(df_final.groupby(["data_previsao", "dia_semana"]).size())
print("\nColunas exportadas:")
print(", ".join(df_final.columns))
print("\nPrecipitação total prevista por data (mm):")
print(df_final.groupby("data_previsao")["precipitacao_mm_3h"].sum(min_count=1))
print("\nValores preenchidos:")
print(df_final[["probabilidade_chuva_pct", "nebulosidade_pct"]].notna().sum())

Registros por data:
data_previsao  dia_semana   
19/09          Sábado           8
20/09          Domingo          8
21/09          Segunda-feira    8
22/09          Terça-feira      8
23/09          Quarta-feira     8
24/09          Quinta-feira     8
25/09          Sexta-feira      8
dtype: int64

Colunas exportadas:
data_previsao, dia_semana, hora, temperatura_c, umidade_pct, direcao_vento, velocidade_vento_kmh, precipitacao_mm_3h, probabilidade_chuva_pct, nebulosidade_pct, coletado_em

Precipitação total prevista por data (mm):
data_previsao
19/09    0.0
20/09    0.0
21/09    0.0
22/09    0.0
23/09    1.5
24/09    3.9
25/09    1.8
Name: precipitacao_mm_3h, dtype: float64

Valores preenchidos:
probabilidade_chuva_pct    55
nebulosidade_pct           56
dtype: int64


In [3]:
display(df_final)

,data_previsao,dia_semana,hora,temperatura_c,umidade_pct,direcao_vento,velocidade_vento_kmh,precipitacao_mm_3h,probabilidade_chuva_pct,nebulosidade_pct,coletado_em
0,19/09,Sábado,00h,21,81,L,16,NaN,NaN,0,2026-09-20 08:26:35
1,19/09,Sábado,03h,19,90,L,16,0.0,0.0,0,2026-09-20 08:26:35
2,19/09,Sábado,06h,18,95,L,16,0.0,0.0,1,2026-09-20 08:26:35
3,19/09,Sábado,09h,22,74,NE,16,0.0,0.0,19,2026-09-20 08:26:35
4,19/09,Sábado,12h,28,51,NE,16,0.0,0.0,49,2026-09-20 08:26:35
5,19/09,Sábado,15h,29,44,NE,16,0.0,0.0,28,2026-09-20 08:26:35
6,19/09,Sábado,18h,26,49,L,16,0.0,0.0,1,2026-09-20 08:26:35
7,19/09,Sábado,21h,22,71,NE,16,0.0,0.0,1,2026-09-20 08:26:35
8,20/09,Domingo,00h,20,77,NE,14,0.0,0.0,0,2026-09-20 08:26:35
9,20/09,Domingo,03h,18,86,NE,14,0.0,0.0,0,2026-09-20 08:26:35


In [4]:
df_final.to_html("previsao_bambui.html", index=False, classes="table table-hover table-striped")


Queimadas

In [8]:
import requests
from bs4 import BeautifulSoup

# URL da página ordenada por data
url = "https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/10min/?C=M;O=D"

# Faz o request da página
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Encontra o primeiro link para CSV
link = soup.find("a", href=lambda href: href and href.endswith(".csv"))
latest_file = link["href"]

# Monta a URL completa
download_url = "https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/10min/" + latest_file

# Baixa o arquivo
csv_data = requests.get(download_url).content
with open(latest_file, "wb") as f:
    f.write(csv_data)

print(f"Arquivo mais recente baixado: {latest_file}")


Arquivo mais recente baixado: focos_10min_20260920_1120.csv


In [9]:
import os
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Diretório para salvar os arquivos
download_dir = "dados_queimadas"

# Cria o diretório se não existir
os.makedirs(download_dir, exist_ok=True)

# Limpa o diretório antes de baixar o novo arquivo
for file in os.listdir(download_dir):
    file_path = os.path.join(download_dir, file)
    if os.path.isfile(file_path):
        os.remove(file_path)

# URL da página ordenada por data
url = "https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/10min/?C=M;O=D"

# Faz o request da página
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Encontra o primeiro link para CSV
link = soup.find("a", href=lambda href: href and href.endswith(".csv"))
latest_file = link["href"]

# Monta a URL completa
download_url = "https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/10min/" + latest_file

# Baixa o arquivo
csv_data = requests.get(download_url).content
file_path = os.path.join(download_dir, latest_file)

with open(file_path, "wb") as f:
    f.write(csv_data)

print(f"Arquivo mais recente baixado: {file_path}")

# Cria um DataFrame com os dados
df = pd.read_csv(file_path)

print("DataFrame criado com sucesso!")
print(df.head())


Arquivo mais recente baixado: dados_queimadas\focos_10min_20260920_1120.csv
DataFrame criado com sucesso!
       lat      lon satelite                 data
0 -12.7428 -46.1830  GOES-19  2026-09-20 11:00:00
1 -12.7621 -46.1800  GOES-19  2026-09-20 11:00:00
2  -7.5334 -45.4367  GOES-19  2026-09-20 11:00:00
3 -10.3168 -50.2352  GOES-19  2026-09-20 11:00:00
